# Day 1: Fisher's LDA from Scratch, Validated Against scikit-learn

**Goal:** show that `fisher_lda.FisherLDA` (pure NumPy) reproduces
`sklearn.discriminant_analysis.LinearDiscriminantAnalysis` on the Dry Bean dataset.

Checks performed:
1. Dataset sanity check
2. Scatter matrices and Fisher's criterion
3. Projections (LD1..LD6) match sklearn up to sign
4. Predictions and probabilities match sklearn
5. Explained variance ratio per discriminant axis
6. First look at the LD1 vs LD2 projection

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from fisher_lda import FisherLDA, load_dry_bean

X, y = load_dry_bean("../data/raw/Dry_Bean_Dataset.xlsx")   # or a .csv from Kaggle
print(X.shape, y.nunique(), "classes")
y.value_counts()

## 1. Sanity check

In [ ]:
print("Missing values:", int(X.isna().sum().sum()))
print("Duplicate rows:", int(X.duplicated().sum()))
X.describe().T[["mean", "std", "min", "max"]]

## 2. Fit our model on a stratified train/test split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X.values, y.values, test_size=0.25, stratify=y, random_state=42)

ours = FisherLDA().fit(X_tr, y_tr)
ref  = LinearDiscriminantAnalysis(solver="svd").fit(X_tr, y_tr)

print("Classes:", ours.classes_)
print("Max discriminant axes:", ours.scalings_.shape[1])
print("Train acc (ours / sklearn):", ours.score(X_tr, y_tr), ref.score(X_tr, y_tr))
print("Test  acc (ours / sklearn):", ours.score(X_te, y_te), ref.score(X_te, y_te))

## 3. Projections match up to sign

In [ ]:
Z_ours, Z_ref = ours.transform(X_te), ref.transform(X_te)
print("Max |abs(diff)| across all axes:", np.abs(np.abs(Z_ours) - np.abs(Z_ref)).max())
assert np.allclose(np.abs(Z_ours), np.abs(Z_ref), atol=1e-6)

## 4. Predictions and probabilities

In [ ]:
agree = (ours.predict(X_te) == ref.predict(X_te)).mean()
proba_diff = np.abs(ours.predict_proba(X_te) - ref.predict_proba(X_te)).max()
print(f"Prediction agreement: {agree:.4%}")
print(f"Max probability difference: {proba_diff:.2e}")

## 5. Explained variance per discriminant axis

In [ ]:
evr = pd.DataFrame({"ours": ours.explained_variance_ratio_, "sklearn": ref.explained_variance_ratio_},
                   index=[f"LD{i+1}" for i in range(len(ref.explained_variance_ratio_))])
display(evr.style.format("{:.4f}"))
evr["ours"].plot.bar(title="Share of between-class separation per axis"); plt.ylabel("ratio"); plt.show()

## 6. First look: LD1 vs LD2

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for c in ours.classes_:
    m = y_te == c
    ax.scatter(Z_ours[m, 0], Z_ours[m, 1], s=8, alpha=.6, label=c)
ax.set_xlabel("LD1"); ax.set_ylabel("LD2"); ax.legend(markerscale=3)
ax.set_title("Dry Beans in discriminant space (test set)")
plt.savefig("../reports/figures/day1_ld1_ld2.png", dpi=150, bbox_inches="tight"); plt.show()

## Takeaways (fill in after running)
- Projections, predictions, probabilities and explained variance match sklearn to ~1e-6.
- Which classes separate along LD1? Which overlap (expect Barbunya / Cali / Sira)?
- How many axes are needed to reach ~95% of the separation?